In [1]:
!python --version

Python 3.8.10


In [2]:
import tensorflow as tf
import larq as lq
import numpy as np
import matplotlib.pyplot as plt


In [3]:
import larq_zoo as lqz

In [4]:
from urllib.request import urlopen
from PIL import Image

In [5]:
img_path = "https://raw.githubusercontent.com/larq/zoo/master/tests/fixtures/elephant.jpg"

with urlopen(img_path) as f:
    img = Image.open(f).resize((224, 224))

x = tf.keras.preprocessing.image.img_to_array(img)
x = lqz.preprocess_input(x)
x = np.expand_dims(x, axis=0)

2026-01-04 10:42:55.577989: W tensorflow/stream_executor/platform/default/dso_loader.cc:55] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-01-04 10:42:55.578004: E tensorflow/stream_executor/cuda/cuda_driver.cc:313] failed call to cuInit: UNKNOWN ERROR (303)
2026-01-04 10:42:55.578011: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (shakti1913): /proc/driver/nvidia/version does not exist
2026-01-04 10:42:55.578160: I tensorflow/core/platform/cpu_feature_guard.cc:143] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 FMA
2026-01-04 10:42:55.581000: I tensorflow/core/platform/profile_utils/cpu_utils.cc:102] CPU Frequency: 2995200000 Hz
2026-01-04 10:42:55.581621: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x732b70000b70 initialized for platform Host (this does not guarantee that

In [6]:
model = lqz.sota.QuickNet(weights="imagenet")
preds = model.predict(x)
lqz.decode_predictions(preds, top=5)[0]

40960/35363 [==================================] - 0s 1us/step


[('n02504458', 'African_elephant', 0.8370481),
 ('n01871265', 'tusker', 0.1613821),
 ('n02504013', 'Indian_elephant', 0.0015687655),
 ('n02410509', 'bison', 3.5993227e-07),
 ('n02408429', 'water_buffalo', 2.7022907e-07)]

In [ ]:
lq.models.summary(model)

In [24]:
from tensorflow.keras import layers, models

base = lqz.sota.QuickNet(
    input_shape=(48, 48, 3),
    weights="imagenet",
    include_top=False,
)
base.trainable = False
from larq_zoo.core import utils

x = base.layers[9].output
x = utils.global_pool(x)
x = lq.layers.QuantDense(
                128,
                kernel_initializer="glorot_normal",
            )(x)
x = lq.layers.QuantDense(
                2,
                kernel_initializer="glorot_normal",
            )(x)
x = tf.keras.layers.Activation("softmax", dtype="float32")(x)

model = models.Model(base.input, x)

In [25]:
lq.models.summary(model)

+model_3 stats------------------------------------------------------------------------------------------------+
| Layer                     Input prec.           Outputs  # 1-bit  # 32-bit  Memory  1-bit MACs  32-bit MACs |
|                                 (bit)                        x 1       x 1    (kB)                          |
+-------------------------------------------------------------------------------------------------------------+
| input_7                             -   (-1, 48, 48, 3)        0         0       0           ?            ? |
| quant_conv2d_126                    -  (-1, 24, 24, 16)        0       432    1.69           0       248832 |
| batch_normalization_132             -  (-1, 24, 24, 16)        0        32    0.12           0            0 |
| activation_27                       -  (-1, 24, 24, 16)        0         0       0           ?            ? |
| quant_depthwise_conv2d_6            -  (-1, 12, 12, 16)        0       144    0.56           0        

In [ ]:
IMAGE_SIZE = (48, 48)
BATCH_SIZE = 32
DATA_DIR = "/home/giri/Downloads/dataset" # Replace with your directory path
VALIDATION_SPLIT = 0.2

data_dir =DATA_DIR
def prep_fn(img):
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) * 2
    return img


    
# 2. Create an ImageDataGenerator instance and specify data augmentation/preprocessing options
# Rescaling pixel values from [0, 255] to [0, 1] is common
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=prep_fn,
    validation_split=0.2 # Example of using a validation split
)

# 3. Use flow_from_directory to create a generator for training data
train_generator = datagen.flow_from_directory(
    directory=data_dir,
    target_size=IMAGE_SIZE, # All images will be resized to 150x150
    batch_size=BATCH_SIZE,
    class_mode='categorical', # "binary" for 2 classes, "categorical" for more
    subset='training' # Specify subset for training
)

# 4. Use flow_from_directory to create a generator for validation data
validation_generator = datagen.flow_from_directory(
    directory=data_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation' # Specify subset for validation
)

model.compile(
    #tf.keras.optimizers.Adam(lr=0.01, decay=0.0001),
    tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

trained_model = model.fit(train_generator,
    batch_size=BATCH_SIZE, 
    epochs=20,
    validation_data=validation_generator,
    shuffle=True
)